# Notebook 4: Model Serving (SPCS + Online Feature Store)

Deploys the fraud model as a REST endpoint on SPCS and benchmarks end-to-end latency: feature lookup from the Online Feature Store + model inference.

---

### Scoring Flow

```
Incoming transaction
  │
  ├── 4 concurrent Online FS REST lookups   (~10-15ms)
  │   (customer, merchant, DPAN, IP velocity)
  │
  ├── Compute derived features inline        (~1ms)
  │   (velocity ratios, merchant concentration, amount deviation)
  │
  └── XGBoost.predict(147 features)          (~105ms)
      │
      ▼
  Fraud probability → approve / flag / block (~130ms total)
```

### Two Scoring Paths

| Path | Latency | When to use |
|---|---|---|
| **Direct HTTP via PrivateLink** | ~130ms p50 | Real-time decisioning — payment gateway needs instant approve/decline |
| SQL service function | ~530ms p50 | Async pipelines, batch re-scoring, internal Snowflake workflows |

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
import time, os, numpy as np, requests

session = get_active_session()
session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql('USE DATABASE FRAUD_DEMO_PROD').collect()
session.sql('USE SCHEMA ML').collect()
session.sql('USE WAREHOUSE FRAUD_OFS_TRAIN_WH').collect()

PAT = os.environ.get('SNOWFLAKE_PAT', '')
if not PAT:
    print('WARNING: SNOWFLAKE_PAT not set')
print('Context: FRAUD_DEMO_PROD.ML')

## 4.1 Deploy Fraud Scoring Service

Deploys the registered model as a SPCS REST endpoint.

In [ ]:
reg = Registry(session=session, database_name='FRAUD_DEMO_PROD', schema_name='ML')
model_ref = reg.get_model('FRAUD_DETECTION_MODEL').version('v1')
print(f'Model: {model_ref.model_name} v{model_ref.version_name}')

try:
    session.sql('DROP SERVICE IF EXISTS FRAUD_DEMO_PROD.ML.FRAUD_SCORING_SERVICE').collect()
except:
    pass

print('Deploying FRAUD_SCORING_SERVICE...')
model_ref.create_service(
    service_name='FRAUD_SCORING_SERVICE',
    service_compute_pool='FRAUD_OFS_CPU_POOL',
    image_build_compute_pool='FRAUD_OFS_CPU_POOL',
    ingress_enabled=False,
    min_instances=1,
    max_instances=2,
    block=True,
)
print('FRAUD_SCORING_SERVICE deployed:')
print('  Compute pool: FRAUD_OFS_CPU_POOL (CPU_X64_XS)')
print('  ingress_enabled: False (private endpoint for production)')
print('  min_instances: 1 (always warm)')

## 4.2 Get Online Feature Store Endpoints

Retrieves the Online FS REST endpoints for use in the scoring service.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, CreationMode
from snowflake.ml.feature_store import online_service as ofs_utils

fs = FeatureStore(
    session=session, database='FRAUD_DEMO_DEV', name='FEATURE_STORE',
    default_warehouse='FRAUD_OFS_TRAIN_WH', creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

status = fs.get_online_service_status()
QUERY_URL = ofs_utils.endpoint_url(status, 'query')
INGEST_URL = ofs_utils.endpoint_url(status, 'ingest')
HEADERS = {'Authorization': f'Snowflake Token="{PAT}"', 'Content-Type': 'application/json'}

print(f'Online FS Query URL:  {QUERY_URL}')
print(f'Online FS Ingest URL: {INGEST_URL}')
print()
print('The scoring service queries these endpoints at inference time.')
print('For production: configure PrivateLink URL (OnlineServiceAccess.PRIVATELINK)')

## 4.3 Latency Benchmark: Online FS + SPCS

Benchmarks the full scoring flow at production volume (60 txn/min):
1. Fetch all 4 entity velocity features from Online FS (concurrent REST calls)
2. Compute derived features inline
3. Score with XGBoost via SPCS

This represents the actual end-to-end latency from the payment gateway's perspective.

In [ ]:
import concurrent.futures

samples = session.sql('''
    SELECT CUSTOMER_ID, MERCHANT_ID, WALLET_DPAN, IP_ADDRESS, PURCHASE_AMOUNT
    FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS SAMPLE (100 ROWS)
''').collect()

svc_info = session.sql("SHOW SERVICES LIKE 'FRAUD_SCORING_SERVICE' IN SCHEMA FRAUD_DEMO_PROD.ML").collect()
dns_name = svc_info[0]['dns_name'] if svc_info else None
SPCS_URL = f'http://{dns_name}:5000/predict'

def fetch_entity_features(row, headers, query_url):
    def q(name, key_col, key_val):
        r = requests.post(f'{query_url}/api/v1/query', headers=headers, json={
            'name': name, 'version': 'V1', 'object_type': 'feature_view',
            'request_rows': [{'entity': {key_col: key_val}}],
        }, timeout=10)
        return r.json().get('rows', [{}])[0] if r.status_code == 200 else {}

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
        f_cust = ex.submit(q, 'FRAUD_CUSTOMER_VELOCITY', 'CUSTOMER_ID', row['CUSTOMER_ID'])
        f_merch = ex.submit(q, 'FRAUD_MERCHANT_VELOCITY', 'MERCHANT_ID', row['MERCHANT_ID'])
        f_dpan  = ex.submit(q, 'FRAUD_DPAN_VELOCITY',     'WALLET_DPAN',  row['WALLET_DPAN'])
        f_ip    = ex.submit(q, 'FRAUD_IP_VELOCITY',       'IP_ADDRESS',   row['IP_ADDRESS'])
    return {**f_cust.result(), **f_merch.result(), **f_dpan.result(), **f_ip.result()}

def compute_derived_features(features, txn_amount):
    num_l1h  = features.get('PURCHASES_NUM_L1H', 0) or 0
    num_l1wk = features.get('PURCHASES_NUM_L1WK', 1) or 1
    amt_l1h  = features.get('PURCHASES_AMT_L1H', 0) or 0
    amt_l1wk = features.get('PURCHASES_AMT_L1WK', 1) or 1
    return {
        'VELOCITY_RATIO_L1H_L1WK': num_l1h / num_l1wk,
        'SPEND_BURST_L1H_L1WK':    amt_l1h / amt_l1wk,
    }

N_WARMUP, N_BENCH = 5, 60
latencies = {'ofs_lookup': [], 'inference': [], 'total': []}

for i in range(N_WARMUP + N_BENCH):
    row = samples[i % len(samples)]

    t0 = time.perf_counter()
    features = fetch_entity_features(row, HEADERS, QUERY_URL)
    ofs_ms = (time.perf_counter() - t0) * 1000

    derived = compute_derived_features(features, row['PURCHASE_AMOUNT'])
    all_features = {**features, **derived}
    feature_values = [all_features.get(k, 0) or 0 for k in sorted(all_features.keys())]

    t1 = time.perf_counter()
    resp = requests.post(SPCS_URL, json={'data': [[i] + feature_values]},
                         headers={'Content-Type': 'application/json'}, timeout=30)
    inf_ms = (time.perf_counter() - t1) * 1000
    total_ms = ofs_ms + inf_ms

    if i >= N_WARMUP:
        latencies['ofs_lookup'].append(ofs_ms)
        latencies['inference'].append(inf_ms)
        latencies['total'].append(total_ms)

print('=' * 60)
print('END-TO-END LATENCY (Online FS lookup + SPCS inference)')
print('=' * 60)
for label, key in [('Online FS lookup (4 entities)', 'ofs_lookup'),
                   ('SPCS inference', 'inference'),
                   ('Total end-to-end', 'total')]:
    arr = np.array(latencies[key])
    print(f'\n  {label}:')
    print(f'    p50: {np.percentile(arr,50):.1f}ms  p95: {np.percentile(arr,95):.1f}ms  p99: {np.percentile(arr,99):.1f}ms')

## 4.4 Production Architecture

For production deployment, the payment gateway calls the scoring service via PrivateLink:

```
Payment Gateway
      │
      ▼
AWS API Gateway (auth, WAF)
      │
      ▼  (PrivateLink)
      │
SPCS Container
      │
      ├── REST → Online FS (PrivateLink endpoint, ~10-15ms)
      └── XGBoost inference (~105ms)
      │
      ▼
Decision (~130ms total)
```

**Key production steps:**
1. Set `ingress_enabled=False` on the service (already done above)
2. Configure PrivateLink between customer VPC and Snowflake
3. Use `OnlineServiceAccess.PRIVATELINK` when initialising the Feature Store in the scoring container
4. Set `SNOWFLAKE_PAT` in AWS Secrets Manager — scoring container reads at runtime

**Next:** NB05 — model monitoring and ROI analysis.